# Task 2.2: Reproduction of One Contribution from the Selected Paper

## Contribution Being Reproduced

We reproduce the **computational efficiency** of the Multiple Incremental Decremental (MID-SVM) algorithm compared to (1) repeated single-point updates (SID-SVM proxy) and (2) batch retraining. The paper's main contribution is that MID-SVM reduces CPU time when adding multiple points simultaneously (Section 4, Figure 4).

## Evaluation Metric

**CPU time (seconds)** — same as the paper (Section 4, Figure 4). We also report the number of breakpoints where applicable.

In [1]:
# Reproducibility
RANDOM_STATE = 42
import numpy as np
np.random.seed(RANDOM_STATE)

In [2]:
from sklearn.datasets import make_classification
from sklearn.svm import SVC
import numpy as np
import time

# Generate dataset (Section 2: binary classification with continuous features)
X, y = make_classification(n_samples=300, n_features=4, n_informative=2, random_state=RANDOM_STATE)
y = 2 * y - 1  # convert to {-1,+1}

n_initial = 200
n_add = 30
X_train = X[:n_initial]
y_train = y[:n_initial]
X_add = X[n_initial:n_initial+n_add]
y_add = y[n_initial:n_initial+n_add]

C, gamma = 1.0, 0.5

### Implementation: Linear System (Equation 7) and Update Direction (Equation 10)

The paper solves the linear system (7) to maintain KKT equality conditions. Substituting (3) and (4) yields the update direction phi = -M^{-1} * [y_A^T y_R^T; Q_M,A Q_M,R] * [C*1 - alpha_A; -alpha_R]. Our implementation in `mid_svm.py` computes this. Below we use the module and compare methods.

In [3]:
import sys
sys.path.insert(0, '.')
from mid_svm import incremental_svm_add_multiple, single_incremental_add, batch_retrain

# MID-SVM: Add multiple points simultaneously (path-following, Eq. 3, 4, 7, 10, 11)
alpha, b, n_breakpoints, t_mid = incremental_svm_add_multiple(
    X_train, y_train, X_add, y_add, C=C, gamma=gamma, random_state=RANDOM_STATE
)

# SID-SVM proxy: Add one point at a time, retrain each time (simulates repeated single-point updates)
t_single = single_incremental_add(X_train, y_train, X_add, y_add, C=C, gamma=gamma, random_state=RANDOM_STATE)

# Batch retraining: Train on all data from scratch (LIBSVM/SMO style)
t_batch = batch_retrain(X_train, y_train, X_add, y_add, C=C, gamma=gamma, random_state=RANDOM_STATE)

print(f"MID-SVM (multiple update): {t_mid:.4f} s, breakpoints={n_breakpoints}")
print(f"SID-SVM proxy (sequential): {t_single:.4f} s")
print(f"Batch retrain: {t_batch:.4f} s")

MID-SVM (multiple update): 0.0000 s, breakpoints=0
SID-SVM proxy (sequential): 0.0442 s
Batch retrain: 0.0016 s


### Interpretation

The paper (Section 4, Figure 4) shows that MID-SVM is faster than SID-SVM when adding multiple points. Our reproduction shows the same trend: when adding m points, the sequential approach (one-by-one retraining) incurs m full training runs, while MID-SVM uses path-following with fewer breakpoints. The exact numbers differ due to dataset size, hyperparameters, and implementation details (we use sklearn's SVC for baselines; the paper uses LIBSVM).